# Do the run-maximizing lineups have a prototype?

`explore_lineups.ipynb` profiled **real** MLB lineups slot-by-slot and asked whether each spot has a
recognizable approach 'prototype'. This notebook runs the **same** toolkit on the lineups the Monte
Carlo optimizer (`optimize_lineups.py`) concluded score the **most runs**, then lays the two side by
side.

The payoff question for the whole project: when you reorder purely to maximize runs, does a consistent
shape emerge across teams — and in particular, does power (barrel%) get pushed **higher** in the order
than real managers put it? No clustering here; the clustering step found no discrete archetypes, so we
just look at the raw approach stats by slot, exactly as `explore_lineups.ipynb` did.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

sns.set_theme(style='whitegrid')

STATS = ['k_pct', 'bb_pct', 'chase_pct', 'whiff_pct', 'barrel_pct']
LABELS = {'k_pct': 'Strikeout %', 'bb_pct': 'Walk %', 'chase_pct': 'Chase %',
          'whiff_pct': 'Whiff %', 'barrel_pct': 'Barrel %'}

opt = pd.read_csv('optimal_lineups_2026.csv')
stats = pd.read_csv('hitter_stats_2026.csv')          # same approach stats as explore_lineups
stats = stats[stats['pa'] >= 150].copy()
print(f'{len(opt)} teams optimized')
opt[['team', 'baseline_rpg', 'best_rpg', 'delta_rpg', 'p_value']].head()

## Explode both lineups to (team, slot, hitter) and attach approach stats

`optimal_lineups_2026.csv` stores each lineup as pipe-joined MLBAM ids in batting order. We unpack the
**optimized** order (`best_ids`) and the **real baseline** order (`baseline_ids`) into slot 1-9 rows,
drop any league-average padding, and merge the five approach stats. Using the baseline the simulator
actually ran keeps the real-vs-optimized comparison on the identical 9-hitter pool per team.

In [ ]:
def explode(id_col, label):
    rows = []
    for _, r in opt.iterrows():
        for slot, bid in enumerate(str(r[id_col]).split('|'), start=1):
            rows.append({'team': r['team'], 'slot': slot, 'batter': bid, 'lineup': label})
    out = pd.DataFrame(rows)
    out = out[out['batter'] != 'None'].copy()          # drop league-average padding
    out['batter'] = out['batter'].astype(int)
    return out.merge(stats[['batter', 'name', 'pa'] + STATS], on='batter', how='inner')

best = explode('best_ids', 'optimized')
real = explode('baseline_ids', 'real')
print(f'optimized rows matched to approach stats: {len(best)} | real: {len(real)}')
best.head()

## Slot profiles of the optimized lineups

The 'prototype' for each spot in the run-maximizing orders — what the average hitter the optimizer put
there looks like. Compare against the identical table for real lineups in `explore_lineups.ipynb`.

In [ ]:
profile = best.groupby('slot')[STATS].mean().round(1)
profile['n'] = best.groupby('slot').size()
profile

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, stat in zip(axes.ravel(), STATS):
    ax.bar(profile.index, profile[stat], color='seagreen')
    ax.set_title(LABELS[stat]); ax.set_xlabel('optimized slot'); ax.set_xticks(range(1, 10))
axes.ravel()[-1].axis('off')
fig.suptitle('Mean approach stat by OPTIMIZED lineup slot (2026, PA>=150)', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## Standardized profile heatmap

Each stat z-scored across the 9 optimized-slot means, so the *shape* of the run-maximizing order is
visible on one scale (red = high for that stat, blue = low).

In [ ]:
z = pd.DataFrame(StandardScaler().fit_transform(profile[STATS]),
                 index=profile.index, columns=[LABELS[s] for s in STATS])
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(z, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax,
            cbar_kws={'label': 'z-score across slots'})
ax.set_ylabel('optimized slot'); ax.set_title('Optimized slot profiles (standardized per stat)')
plt.tight_layout(); plt.show()

## Spread within each optimized slot

Means hide dispersion. If the boxes overlap heavily across slots, the optimized order has no real
prototype either — the optimizer is mostly indifferent to who bats where.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, stat in zip(axes.ravel(), STATS):
    sns.boxplot(data=best, x='slot', y=stat, ax=ax, color='darkseagreen')
    ax.set_title(LABELS[stat]); ax.set_xlabel('optimized slot'); ax.set_ylabel('')
axes.ravel()[-1].axis('off')
plt.tight_layout(); plt.show()

## Real vs. optimized: does the optimizer move power up?

Overlay the mean stat-by-slot curve for the real baseline and the optimized order. The hypothesis
predicts the optimized **barrel%** (and K%/whiff%, which travel with power) peaks *earlier* than the
real one.

In [ ]:
both = pd.concat([real, best], ignore_index=True)
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, stat in zip(axes.ravel(), STATS):
    for label, color in [('real', 'steelblue'), ('optimized', 'seagreen')]:
        m = both[both['lineup'] == label].groupby('slot')[stat].mean()
        ax.plot(m.index, m.values, 'o-', color=color, label=label)
    ax.set_title(LABELS[stat]); ax.set_xlabel('slot'); ax.set_xticks(range(1, 10)); ax.legend()
axes.ravel()[-1].axis('off')
fig.suptitle('Real vs. run-maximizing lineups: approach stat by slot', y=1.02, fontsize=13)
plt.tight_layout(); plt.show()

## How distinct / directional are the optimized slots?

The same separation reads as `explore_lineups.ipynb`, computed for both lineups so they can be
compared directly:
- **eta²** = share of a stat's variance explained by slot (0 = slot tells you nothing).
- **Spearman r vs slot** = strength/direction of a monotonic gradient down the order (negative = the
  stat is concentrated in the *top* slots).
- **silhouette** of the 9 slot labels on the standardized 5-stat space (near 0 => slots overlap).

In [ ]:
def separation(df):
    rows = []
    for stat in STATS:
        groups = [g[stat].values for _, g in df.groupby('slot')]
        grand = df[stat].mean()
        eta2 = sum(len(g) * (g.mean() - grand) ** 2 for g in groups) / ((df[stat] - grand) ** 2).sum()
        f, p = f_oneway(*groups)
        rho, _ = spearmanr(df['slot'], df[stat])
        rows.append([LABELS[stat], round(eta2, 3), round(p, 4), round(rho, 3)])
    return pd.DataFrame(rows, columns=['stat', 'eta2', 'ANOVA p', 'Spearman r vs slot'])

for label, df in [('REAL', real), ('OPTIMIZED', best)]:
    print(f'=== {label} ===')
    print(separation(df).to_string(index=False))
    sil = silhouette_score(StandardScaler().fit_transform(df[STATS]), df['slot'])
    print(f'silhouette of 9 slot labels: {sil:.3f}\n')

## Where do the big bats end up?

The crux of the hypothesis: in the run-maximizing orders, what slot do the highest-barrel hitters get
moved to, and is that higher than where they really bat?

In [ ]:
big = (best[['team', 'name', 'batter', 'slot', 'barrel_pct', 'k_pct', 'bb_pct']]
       .merge(real[['batter', 'slot']].rename(columns={'slot': 'real_slot'}), on='batter', how='left')
       .rename(columns={'slot': 'opt_slot'}))
print('Top 15 barrel% hitters: real slot vs optimized slot')
print(big.nlargest(15, 'barrel_pct')[['name', 'team', 'barrel_pct', 'real_slot', 'opt_slot']].to_string(index=False))
print(f"\nmean optimized slot of top-30 barrel hitters: {big.nlargest(30, 'barrel_pct')['opt_slot'].mean():.2f}"
      f"  vs real: {big.nlargest(30, 'barrel_pct')['real_slot'].mean():.2f}")

## Takeaways

_(fill in after running)_

- Does an optimized 'prototype' exist, or do the boxplots/eta²/silhouette say the slots overlap as much
  as they did for real lineups (and for the clusters)?
- Compare the **Spearman r vs slot** for barrel% between real and optimized: a more negative value under
  'optimized' is direct evidence the run-maximizer wants power *higher* in the order.
- Reconcile with the run gains in `optimal_lineups_2026.csv`: if the deltas are tiny, any slot pattern is
  a weak tendency, not a strong rule (order effects are small and PAs are modelled independently).